# Adicionar modelo de resposta manualmente

Notebook **separado** de `ler_dados_pst.ipynb` — aquele lê o `.pst` exportado
do Outlook inteiro; este serve para o caso menor: acrescentar (ou corrigir)
UM modelo de resposta direto em `modelos_resposta_chunks.xlsx`, sem precisar
reexportar e reprocessar a caixa de e-mail inteira.

Gera linhas no MESMO formato que `ler_dados_pst.ipynb` produziria — mesma
regra de chunking (`dividir`, `CHUNK_SIZE=1200`), mesmo cabeçalho
("Modelo de resposta padrao do suporte..."), mesmo `modelo_id` sequencial
(`MODxxxx`) — para que o loader de ingestão
(`app/ingestion/loaders/xlsx_modelos_resposta.py`) não veja diferença entre
uma linha vinda do export e uma escrita aqui.

**Fluxo:**
1. Rode as células até "estado atual do arquivo" para ver o que já existe.
2. Edite a célula `NOVOS_MODELOS` — um dict por modelo novo.
3. Rode o resto: valida, gera as linhas, confira a prévia, faz backup do
   arquivo atual e sobrescreve `modelos_resposta_chunks.xlsx`.
4. Fora do notebook: `python -m scripts.ingest email_modelos`.

**Atenção — reingestão é do arquivo inteiro, não incremental**: `ingest_file`
apaga por `source_path` antes de reindexar (é o que mantém a reingestão
idempotente mesmo quando um arquivo encolhe — ver `app/ingestion/pipeline.py`).
Rodar o passo 4 reprocessa TODOS os modelos do arquivo, não só o que este
notebook acabou de adicionar. Local e sem custo de API (o embedding roda no
HuggingFace local do projeto), então é rápido, mas vale saber.

**Duplicação deliberada, não descuido**: a função `dividir()` logo abaixo é
uma CÓPIA da de `ler_dados_pst.ipynb` (seção 6a), não um import — notebook
não é módulo importável sem ferramenta extra (`nbimporter`), que este projeto
não tem só por causa de uma função. Se `CHUNK_SIZE`/`CHUNK_OVERLAP`/`dividir`
mudarem lá, replique aqui à mão — senão os dois arquivos passam a chunkar
diferente sem que nada avise.

In [29]:
from __future__ import annotations

import shutil
from datetime import datetime
from pathlib import Path

import pandas as pd

# Mesmo arquivo que a ingestão lê (data/raw/<assunto>/ = pasta que
# `python -m scripts.ingest <assunto>` processa — ver app/ingestion/pipeline.py).
CAMINHO_XLSX = Path("../../data/raw/email_modelos/modelos_resposta_chunks.xlsx").resolve()

# FORA de data/raw/ de propósito: `ingest_assunto` varre a pasta do assunto
# inteira com `rglob("*")` — recursivo, sem exceção para pasta com nome
# começando em ponto. Um backup salvo DENTRO de data/raw/email_modelos/
# seria descoberto como arquivo novo na próxima ingestão e voltaria a ser
# indexado, inclusive conteúdo já removido/corrigido numa edição anterior.
PASTA_BACKUP = Path(".bak").resolve()

assert CAMINHO_XLSX.exists(), f"não encontrado: {CAMINHO_XLSX}"
print("arquivo:", CAMINHO_XLSX)

arquivo: C:\Users\henrique.cordeiro\Desktop\PPP\ia-agent-puc-digital\data\raw\email_modelos\modelos_resposta_chunks.xlsx


## A mesma lógica de chunking de `ler_dados_pst.ipynb` (seção 6a)

In [30]:
CHUNK_SIZE      = 1200   # caracteres por chunk (~300 tokens)
CHUNK_OVERLAP   = 150
MIN_CHARS_CHUNK = 80     # descarta ruído


def dividir(texto, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    texto = (texto or "").strip()
    if len(texto) <= size:
        return [texto] if texto else []
    chunks, ini = [], 0
    while ini < len(texto):
        fim = min(ini + size, len(texto))
        if fim < len(texto):
            meio = ini + size // 2
            corte = max(texto.rfind("\n\n", meio, fim),
                        texto.rfind(". ", meio, fim),
                        texto.rfind("\n", meio, fim))
            if corte > ini:
                fim = corte + 1
        chunks.append(texto[ini:fim].strip())
        if fim >= len(texto):
            break
        ini = max(fim - overlap, ini + 1)
    return [c for c in chunks if len(c) >= MIN_CHARS_CHUNK]

## Estado atual do arquivo

In [31]:
xl = pd.ExcelFile(CAMINHO_XLSX)
df_modelos = xl.parse("modelos")
df_chunks = xl.parse("chunks")

print(f"modelos hoje: {len(df_modelos)} | chunks hoje: {len(df_chunks)}")

# "MOD0001" -> 1. Fatia fixa (prefixo "MOD" tem 3 letras) em vez de
# str.removeprefix: funciona em qualquer versão do pandas instalada.
_max_num = df_modelos["modelo_id"].str.slice(3).astype(int).max() if len(df_modelos) else 0
print(f"próximo modelo_id livre: MOD{_max_num + 1:04d}")

df_modelos[["modelo_id", "assunto", "n_ocorrencias", "redirecionado"]].tail(5)

modelos hoje: 183 | chunks hoje: 202
próximo modelo_id livre: MOD0184


,modelo_id,assunto,n_ocorrencias,redirecionado
178,MOD0179,RES: RES: RES: RES: RES: DP pós gerenciamento ...,3,False
179,MOD0180,Atestado de matrícula - [RA],3,False
180,MOD0181,RES: Declaração de estudante/ matrícula,3,False
181,MOD0182,Duvida sobre segunda chamada de prova,1,False
182,MOD0183,Duvida sobre segunda chamada de prova,1,False


## Edite aqui: os modelos novos

Um dict por modelo em `NOVOS_MODELOS`. Campos:

- **`assunto`** (obrigatório) — assunto típico do e-mail que este modelo responde.
- **`texto_modelo`** (obrigatório) — o corpo da resposta. Use `{{campo}}` para
  o que varia por aluno (mesma convenção do resto da base — veja exemplos em
  `df_modelos["texto_modelo"]` na célula acima).
- `slots` (opcional) — nomes dos `{{campo}}` usados, separados por `"; "`
  (ex.: `"nome; valor"`). `None`/omitido se o modelo não tem campo variável.
- `assuntos_variantes` (opcional) — outras formas do mesmo assunto
  (`ENC:`/`RE:`/etc.) que este modelo também responde. Default: só o próprio
  `assunto`.
- `redirecionado` (opcional, default `False`) — `True` se a resposta só
  encaminha o aluno para outro contato, sem resolver aqui.
- `n_ocorrencias` / `n_threads` (opcional, default `1`) — no export real isso
  é frequência MEDIDA no histórico de e-mails. Um modelo escrito à mão não tem
  esse histórico: `1` é o valor honesto ("usado 1x, nesta inserção"). Não
  infle — esse número pode virar sinal de confiança em ranking futuro.

In [32]:
NOVOS_MODELOS = [
    # Apague este exemplo e escreva o(s) seu(s) — um dict por modelo.
    {
        "assunto": "Duvida sobre segunda Autenticador Microsoft",
        "slots": "nome; data",
       "texto_modelo": (
            "Olá, {{nome}}!\n\n"
            "Identificamos que você está com problemas no Microsoft Authenticator e, por isso, não está recebendo o código de autenticação.\n\n"
            "Nesse caso, será necessário entrar em contato com o PUC Digital pelo e-mail puc.digital@puc-campinas.edu.br ou pelo WhatsApp (19) 99689-1420, solicitando o RESET MFA para que sua conta possa ser vinculada novamente ao Microsoft Authenticator em seu dispositivo móvel.\n\n"
            "Após a realização do RESET MFA, será possível configurar novamente o Microsoft Authenticator para realizar o acesso.\n\n"
            "Agradecemos a compreensão.\n\n"
        ),
        # Opcionais — pode remover a chave se não usar, os defaults cobrem:
        "assuntos_variantes": None,   # None -> vira só [assunto]
        "redirecionado": False,
        "n_ocorrencias": 1,
        "n_threads": 1,
    },
]

## Validação

In [33]:
_existentes = set(
    pd.concat([df_modelos["assunto"], df_modelos["assuntos_variantes"]])
    .dropna().str.casefold()
)

# Campos que podem vir como string multi-linha entre parênteses — é aí que
# mora o erro mais comum ao editar NOVOS_MODELOS: uma vírgula sobrando depois
# da última linha do texto (antes do `)`) faz o Python ler aquilo como uma
# TUPLA de 1 item, não como a string concatenada. `.strip()` numa tupla dá
# `AttributeError: 'tuple' object has no attribute 'strip'` — mensagem que
# não diz a causa. Checar o tipo aqui, ANTES de chamar `.strip()`, troca esse
# erro cru por um diagnóstico direto.
_CAMPOS_TEXTO = ("assunto", "texto_modelo", "slots", "assuntos_variantes")

for i, m in enumerate(NOVOS_MODELOS):
    rotulo = f"NOVOS_MODELOS[{i}]"
    for campo in _CAMPOS_TEXTO:
        valor = m.get(campo)
        if valor is not None and not isinstance(valor, str):
            raise TypeError(
                f"{rotulo}['{campo}'] é {type(valor).__name__} ({valor!r}), esperava string ou None. "
                "Causa comum: uma vírgula sobrando dentro dos parênteses de uma string "
                "multi-linha vira uma tupla de 1 item — confira se não tem `,` depois "
                "da última linha do texto, logo antes do `)`."
            )
    assert (m.get("assunto") or "").strip(), f"{rotulo}: 'assunto' vazio"
    assert (m.get("texto_modelo") or "").strip(), f"{rotulo}: 'texto_modelo' vazio"
    if m["assunto"].casefold() in _existentes:
        print(f"AVISO {rotulo}: já existe um modelo com assunto parecido — "
              f"confira se não é duplicata antes de seguir: {m['assunto']!r}")

print(f"{len(NOVOS_MODELOS)} modelo(s) validado(s) — sem erro bloqueante.")

1 modelo(s) validado(s) — sem erro bloqueante.


## Gera as linhas (mesma lógica de `ler_dados_pst.ipynb`, seção 6a)

In [34]:
agora = pd.Timestamp(datetime.now())
proximo_num = _max_num + 1

novas_linhas_modelos = []
novas_linhas_chunks = []

for m in NOVOS_MODELOS:
    modelo_id = f"MOD{proximo_num:04d}"
    proximo_num += 1

    assunto = m["assunto"].strip()
    slots = (m.get("slots") or "").strip() or None
    texto_modelo = m["texto_modelo"].strip()
    assuntos_variantes = m.get("assuntos_variantes") or assunto
    redirecionado = bool(m.get("redirecionado", False))
    n_ocorrencias = int(m.get("n_ocorrencias", 1))
    n_threads = int(m.get("n_threads", 1))

    novas_linhas_modelos.append({
        "modelo_id": modelo_id, "assunto": assunto, "n_ocorrencias": n_ocorrencias,
        "n_threads": n_threads, "primeira_data": agora, "ultima_data": agora,
        "slots": slots, "assuntos_variantes": assuntos_variantes,
        "thread_ids": "",  # sem thread real: modelo escrito à mão, não extraído de e-mail
        "texto_modelo": texto_modelo, "redirecionado": redirecionado,
    })

    # --- a partir daqui, a MESMA lógica de ler_dados_pst.ipynb, seção 6a ---
    cabec = (f"Modelo de resposta padrao do suporte "
             f"(usado {n_ocorrencias}x em {n_threads} thread{'s' if n_threads != 1 else ''}).")
    corpo = (f"Assunto tipico: {assunto}\n"
             + (f"Campos a preencher: {slots}\n" if slots else "")
             + f"\n{texto_modelo}")
    ano = agora.year

    for i, ch in enumerate(dividir(corpo), start=1):
        texto = f"{cabec}\n\n{ch}"
        novas_linhas_chunks.append({
            "chunk_id": f"{modelo_id}-{i:03d}", "modelo_id": modelo_id, "tipo": "modelo",
            "parte": i, "assunto": assunto, "assuntos_variantes": assuntos_variantes,
            "slots": slots, "redirecionado": redirecionado,
            "n_ocorrencias": n_ocorrencias, "n_threads": n_threads,
            "primeira_data": agora, "ultima_data": agora, "ano": ano,
            "n_chars": len(texto), "tokens_aprox": round(len(texto) / 4), "texto": texto,
        })

print(f"{len(novas_linhas_modelos)} modelo(s) -> {len(novas_linhas_chunks)} chunk(s) novo(s)")
pd.DataFrame(novas_linhas_chunks)[["chunk_id", "parte", "n_chars", "tokens_aprox"]]

1 modelo(s) -> 1 chunk(s) novo(s)


,chunk_id,parte,n_chars,tokens_aprox
0,MOD0184-001,1,711,178


## Confira antes de gravar

In [35]:
for linha in novas_linhas_chunks:
    print("=" * 70)
    print(linha["chunk_id"], "-", linha["n_chars"], "chars,", linha["tokens_aprox"], "tokens aprox.")
    print(linha["texto"])

MOD0184-001 - 711 chars, 178 tokens aprox.
Modelo de resposta padrao do suporte (usado 1x em 1 thread).

Assunto tipico: Duvida sobre segunda Autenticador Microsoft
Campos a preencher: nome; data

Olá, {{nome}}!

Identificamos que você está com problemas no Microsoft Authenticator e, por isso, não está recebendo o código de autenticação.

Nesse caso, será necessário entrar em contato com o PUC Digital pelo e-mail puc.digital@puc-campinas.edu.br ou pelo WhatsApp (19) 99689-1420, solicitando o RESET MFA para que sua conta possa ser vinculada novamente ao Microsoft Authenticator em seu dispositivo móvel.

Após a realização do RESET MFA, será possível configurar novamente o Microsoft Authenticator para realizar o acesso.

Agradecemos a compreensão.


## Grava (com backup)

Sobrescreve `modelos_resposta_chunks.xlsx` com o conteúdo antigo + as linhas
novas. Backup do arquivo anterior vai para `.bak/` (ao lado deste notebook,
fora de `data/raw/` — ver comentário na 1ª célula de código).

In [36]:
PASTA_BACKUP.mkdir(exist_ok=True)
carimbo = datetime.now().strftime("%Y%m%dT%H%M%S")
destino_backup = PASTA_BACKUP / f"{CAMINHO_XLSX.stem}_{carimbo}{CAMINHO_XLSX.suffix}"
shutil.copy2(CAMINHO_XLSX, destino_backup)
print("backup:", destino_backup)

df_modelos_final = pd.concat([df_modelos, pd.DataFrame(novas_linhas_modelos)], ignore_index=True)
df_chunks_final = pd.concat([df_chunks, pd.DataFrame(novas_linhas_chunks)], ignore_index=True)

# Mesma ordem de coluna do arquivo original — o loader de ingestão não depende
# disso, mas mantém previsível para quem abrir no Excel.
df_modelos_final = df_modelos_final[df_modelos.columns]
df_chunks_final = df_chunks_final[df_chunks.columns]

with pd.ExcelWriter(CAMINHO_XLSX, engine="openpyxl") as xw:
    df_chunks_final.to_excel(xw, sheet_name="chunks", index=False)
    df_modelos_final.to_excel(xw, sheet_name="modelos", index=False)

print(f"gravado: {CAMINHO_XLSX}")
print(f"modelos: {len(df_modelos)} -> {len(df_modelos_final)}")
print(f"chunks : {len(df_chunks)} -> {len(df_chunks_final)}")

backup: C:\Users\henrique.cordeiro\Desktop\PPP\ia-agent-puc-digital\scripts\Ler_dados_exportados_email\.bak\modelos_resposta_chunks_20260827T155001.xlsx
gravado: C:\Users\henrique.cordeiro\Desktop\PPP\ia-agent-puc-digital\data\raw\email_modelos\modelos_resposta_chunks.xlsx
modelos: 183 -> 184
chunks : 202 -> 203


## Próximo passo (fora do notebook)

```bash
python -m scripts.ingest email_modelos
```

Reindexa o arquivo inteiro (ver aviso no topo — não é incremental). Se algo
saiu errado, o arquivo anterior está em `.bak/` ao lado deste notebook.